# Environment and Setup

In [ ]:
!pip install matplotlib

In [2]:
import numpy as np
import pandas as pd
from geopy.distance import distance
from shapely.geometry import LineString, Point
from pyproj import Transformer
import ast

from sklearn.cluster import DBSCAN

import scipy
import json
import zipfile
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.spatial import cKDTree
from datetime import datetime, timedelta
from collections import defaultdict

# Recommendation Thresholds

In [ ]:
# Carpool thresholds
MAX_DISTANCE = 2 # miles
MAX_TIME_DIFF = 30 # minutes

#Tunable input parameters
WALK_RADIUS_MI = 0.33
# Maximum walk distance (mi) to a stop for it to be considered "nearby"
# 530 m ≈ 6-7 minutes walking at average pace
# Increase if in walkable corridors

TIME_WINDOW_MINUTES = 30
# How many minutes either side of the user's typical departure time we'll consider a transit departure "aligned"
# ±30 min = 1-hour window total. Wider = more options, less precision

WALK_SPEED_MI_H = 3.0
# Average walking speed in mi/h 
# Used to estimate walk time from distance

AVG_DRIVE_SPEED_MI_H = 23.0
# Average driving speed in Atlanta (mi/h) used to estimate drive time
# for comparison. The simulation uses 30; using 23 to
# account for city traffic

MAX_TRANSFERS = 1
# Maximum number of transfers (route changes) to allow in a transit option
# 0 = direct only, 1 = one connection. 

TOP_N_TRANSIT = 2
# Number of transit alternatives to return per recurring trip.
# Showing 2 options (e.g. direct bus + rail option)

## IS THIS HOW WE DID IT THO??? OR ARE WE ALLOWING ONE OFFS ?? ##
MIN_TRIP_OCCURRENCES = 3
# Same value as in the carpool matching notebook
# A trip must recur at least this many times to be eligible for suggestions
# This prevents surfacing transit options for one-off errands

# CO2 emission factors
CO2_DRIVE_LB_PER_MI  = 0.745079   # Average US passenger car: (~0.21 kg/km) 
CO2_BUS_LB_PER_MI    = 0.31577166  # MARTA bus per passenger: (~0.089 kg/km) 
CO2_RAIL_LB_PER_MI   = 0.14546784  # MARTA rail per passenger: (~0.041 kg/km) 
# Sources: EPA (2023), MARTA sustainability reports

#For matching algorithm
#max distance away for DBSCAN clsuter
DBSCAN_BUFFER_MILES = 1
#min number of trips for DBSCAN cluster
DBSCAN_MIN_SAMPLES = 2

print('Configuration loaded.')
print(f'Walk radius: {WALK_RADIUS_MI} mi')
print(f'Time window: ±{TIME_WINDOW_MINUTES} min')
print(f'Max transfers: {MAX_TRANSFERS}')
print(f'Min trip occurrences: {MIN_TRIP_OCCURRENCES}')
print(f'Results per trip: {TOP_N_TRANSIT}')

# Load Data

In [ ]:
#GTFS data
calendar_dates = pd.read_csv('/Users/allandakriener/Desktop/MARTA/calendar_dates.txt')
calendar = pd.read_csv('/Users/allandakriener/Desktop/MARTA/calendar.txt')
routes = pd.read_csv('/Users/allandakriener/Desktop/MARTA/routes.txt')
shapes = pd.read_csv('/Users/allandakriener/Desktop/MARTA/shapes.txt')
stop_times = pd.read_csv('/Users/allandakriener/Desktop/MARTA/stop_times.txt')
stops = pd.read_csv('/Users/allandakriener/Desktop/MARTA/stops.txt')
trips = pd.read_csv('/Users/allandakriener/Desktop/MARTA/trips.txt')

TRIPS_PATH = '../data/simulated/trips.csv'
USERS_PATH = '../data/simulated/users.csv'
FRIENDS_PATH = '../data/simulated/friendships.csv'

trips = pd.read_csv(TRIPS_PATH)
users = pd.read_csv(USERS_PATH)
friendships = pd.read_csv(FRIENDS_PATH)

print(f'Loaded {len(trips):,} trips across {trips["user_id"].nunique()} users')
print(f'Loaded {len(users)} user profiles')
print(f'Loaded {len(friendships)} friendship edges')

# Friendship Parsing

In [ ]:
def parse_friendships(friendships):
    bidirectional = pd.concat([
        friendships.rename(columns={'user_id_1': 'user', 'user_id_2': 'friend'}),
        friendships.rename(columns={'user_id_2': 'user', 'user_id_1': 'friend'})
    ])

    friendship_dict = bidirectional.groupby('user')['friend'].apply(set).to_dict()
    return friendship_dict

# Recurring Trip Detection

In [ ]:
def get_location_clusters(trips, buffer_miles: int = 1, min_samples: int = 2) -> pd.DataFrame:
    trips = trips.copy()
    for col, lat_col, lon_col in [
        ('start_cluster', 'start_lat', 'start_lon'),
        ('end_cluster', 'end_lat', 'end_lon')
    ]:
        coords = trips[[lat_col, lon_col]].values
        coords_rad = np.radians(coords)
        
        # eps in radians: buffer_miles / earth_radius
        eps = buffer_miles / 3958.8
        
        labels = DBSCAN(eps=eps, min_samples=min_samples, algorithm='ball_tree', metric='haversine').fit_predict(coords_rad)
        trips[col] = labels
    
    return trips

def get_recurring_trips(trips, min_occurrences=2):
    trips = get_location_clusters(trips)

    # drop trips where either endpoint was an outlier
    trips = trips[
        (trips['start_cluster'] != -1) &
        (trips['end_cluster'] != -1)
    ]

    # count occurrences of each OD cluster pair
    od_counts = (
        trips
        .groupby(['start_cluster', 'end_cluster'])
        .size()
        .reset_index(name='occurrences')
    )

    recurring_od = od_counts[od_counts['occurrences'] >= min_occurrences][['start_cluster', 'end_cluster', 'occurrences']]

    return trips.merge(recurring_od, on=['start_cluster', 'end_cluster'], how='inner')

# Carpool Engine and Helpers

In [ ]:
def geodesic(a, b):
    dist_miles = distance(a, b).miles
    return dist_miles

def preprocess_trips(trips):
    #type conversion
    trips = trips.copy()
    trips['start_time'] = pd.to_datetime(trips['start_time'], format='mixed')
    trips['end_time'] = pd.to_datetime(trips['end_time'], format='mixed')
    if isinstance(trips['route_points'].iloc[0], str):
        trips['route_points'] = trips['route_points'].apply(ast.literal_eval)
    
    # feature engineering for recurring trip detection
    trips['time_bucket'] = trips['start_time'].dt.floor('30min').dt.time
    trips['day_of_week'] = trips['start_time'].dt.dayofweek
    return trips

def get_utm_transformer(lat, lon):
    # get the appropriate UTM zone transformer for a given location
    utm_zone = int((lon + 180) / 6) + 1
    hemisphere = "north" if lat >= 0 else "south"
    epsg = 32600 + utm_zone if hemisphere == "north" else 32700 + utm_zone
    return Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)

def project_and_create_polyline(route_points, transformer):
    route_proj = [transformer.transform(lon, lat) for lat, lon in route_points]
    return LineString(route_proj)

def get_pickup_begin_score(my_trip, other_trip, max_distance):
    # how close other trip's start is to my trip's start
    my_start = (my_trip['start_lat'], my_trip['start_lon'])
    other_start = (other_trip['start_lat'], other_trip['start_lon'])
    distance = geodesic(my_start, other_start)
    pickup_score = 1 - min(distance / max_distance, 1)
    return pickup_score

def get_pickup_enroute_score(my_route, other_trip, transformer, max_distance):
    # how close other trip's start is to any of my trip's route points
    METERS_TO_MILES = 0.000621371
    other_start = Point(transformer.transform(other_trip['start_lon'], other_trip['start_lat']))
    distance = other_start.distance(my_route) * METERS_TO_MILES
    pickup_score = 1 - min(distance / max_distance, 1)
    return pickup_score

def get_dropoff_end_score(my_trip, other_trip, max_distance):
    # how close other trip's end is to my trip's end
    my_end = (my_trip['end_lat'], my_trip['end_lon'])
    other_end = (other_trip['end_lat'], other_trip['end_lon'])
    distance = geodesic(my_end, other_end)
    dropoff_score = 1 - min(distance / max_distance, 1)
    return dropoff_score

def get_dropoff_enroute_score(my_route, other_trip, transformer, max_distance):
    # how close other trip's end is to any of my trip's route points
    METERS_TO_MILES = 0.000621371
    other_end = Point(transformer.transform(other_trip['end_lon'], other_trip['end_lat']))
    distance = other_end.distance(my_route) * METERS_TO_MILES
    dropoff_score = 1 - min(distance / max_distance, 1)
    return dropoff_score

def get_start_time_score(my_trip, other_trip, max_time_diff):
    time_diff = abs(my_trip['start_time'] - other_trip['start_time']).total_seconds() / 60
    time_score = 1 - min(time_diff / max_time_diff, 1)
    return time_score

def get_end_time_score(my_trip, other_trip, max_time_diff):
    time_diff = abs(my_trip['end_time'] - other_trip['end_time']).total_seconds() / 60
    time_score = 1 - min(time_diff / max_time_diff, 1)
    return time_score


def score_trips_for_user(user_id, trips, max_distance, max_time_diff, test=False, verbose=False):
    trips = preprocess_trips(trips)
    my_trips = trips[trips['user_id'] == user_id]
    my_recurring_trips = get_recurring_trips(my_trips)

    if my_recurring_trips.empty:
        print("No recurring trips found for user", user_id)
        return []

    my_friends = parse_friendships(friendships).get(user_id, set())
    friend_trips = [get_recurring_trips(trips[trips['user_id'] == friend]) for friend in my_friends]
    friend_trips = [df for df in friend_trips if not df.empty]
    other_recurring_trips = pd.concat(friend_trips) if friend_trips else pd.DataFrame()

    if other_recurring_trips.empty:
        print("No recurring trips found for friends of user", user_id)
        return []

    scores = []
    for _, my_trip in my_recurring_trips.iterrows():
        if verbose:
            print("Scoring trip:", my_trip['trip_id'])
            
        transformer = get_utm_transformer(my_trip['start_lat'], my_trip['start_lon'])
        my_route = project_and_create_polyline(my_trip['route_points'], transformer)

        for i, other_trip in other_recurring_trips.iterrows():
            pickup_begin_score = get_pickup_begin_score(my_trip, other_trip, max_distance)
            dropoff_end_score = get_dropoff_end_score(my_trip, other_trip, max_distance)
            
            pickup_enroute_score = get_pickup_enroute_score(my_route, other_trip, transformer, max_distance)
            if pickup_begin_score == 0 and pickup_enroute_score == 0:
                # no good pickup options, so skip this trip
                continue
            dropoff_enroute_score = get_dropoff_enroute_score(my_route, other_trip, transformer, max_distance)
            if dropoff_end_score == 0 and dropoff_enroute_score == 0:
                # no good dropoff options, so skip this trip
                continue

            start_time_score = get_start_time_score(my_trip, other_trip, max_time_diff)
            end_time_score = get_end_time_score(my_trip, other_trip, max_time_diff)

            # can pickup and dropoff at end, or pickup enroute and dropoff at end, or pickup at beginning and dropoff enroute, or pickup and dropoff enroute
            # time alignment should match with pickup/dropoff alignment - if pickup is enroute, then start time should be less important, if dropoff is enroute, then end time should be less important
            route_score = max(pickup_begin_score + dropoff_end_score + 0.5*start_time_score + 0.5*end_time_score,
                              pickup_enroute_score + dropoff_end_score + 0.25*start_time_score + 0.75*end_time_score,
                              pickup_begin_score + dropoff_enroute_score + 0.75*start_time_score + 0.25*end_time_score,
                              pickup_enroute_score + dropoff_enroute_score + 0.5*start_time_score + 0.5*end_time_score)

            
            total_score = route_score / 3
            scores.append({
                'user_id': user_id,
                'other_user': other_trip['user_id'],
                'my_trip': my_trip['trip_id'],
                'other_trip': other_trip['trip_id'],
                'score': round(total_score, 6)
            })

        if test:
            break
    
    scores.sort(key=lambda x: x['score'], reverse=True)
    return scores

# Transit Engine and Helpers

In [ ]:
#Cast stop coordinates to float
stops['stop_lat'] = stops['stop_lat'].astype(float)
stops['stop_lon'] = stops['stop_lon'].astype(float)

print(f'GTFS loaded: {len(stops):,} stops, {len(routes):,} routes')

#Preprocess simulated user trip data
def preprocess_trips(trips):
    trips['start_time'] = pd.to_datetime(trips['start_time'], format='mixed')
    trips['end_time'] = pd.to_datetime(trips['end_time'], format='mixed')
    if type(trips['route_points'].iloc[0]) == str:
        trips['route_points'] = trips['route_points'].apply(ast.literal_eval)
    return trips

#calculate distance between two points
def distance_calc(point1, point2):
    return distance(point1, point2).miles

#store transit type information for each stop
def build_stop_mode_lookup(stops, stop_times, trips, routes):
    st = stop_times[['stop_id', 'trip_id']].drop_duplicates()
    tr = trips[['trip_id', 'route_id']].drop_duplicates()
    rt = routes[['route_id', 'route_type']].drop_duplicates()

    #join stop -> trip -> route
    stop_route_types = (
        st.merge(tr, on='trip_id', how='left')
          .merge(rt, on='route_id', how='left')
          [['stop_id', 'route_type']]
          .drop_duplicates())

    #aggregate route types seen at each stop
    stop_modes = (
        stop_route_types.groupby('stop_id')['route_type']
        .apply(lambda x: sorted(set(x.dropna())))
        .reset_index(name='route_types'))

    def classify(route_types):
        route_types = set(route_types)

        has_bus = 3 in route_types
        has_rail = any(rt in route_types for rt in [0, 1, 2])

        if has_bus and has_rail:
            return 'both'
        elif has_rail:
            return 'rail'
        elif has_bus:
            return 'bus'
        else:
            return 'unknown'

    stop_modes['mode'] = stop_modes['route_types'].apply(classify)
    return stop_modes

#find stops to users start and end points
def find_nearest_stop(point, stops):
    min_distance = float('inf')
    nearest_stop_id = None
    for idx, stop in stops.iterrows():
        stop_point = (stop['stop_lat'], stop['stop_lon'])
        dist = distance_calc(point, stop_point)
        if dist < min_distance:
            min_distance = dist
            nearest_stop_id = stop['stop_id']
    return nearest_stop_id

stop_mode_lookup = build_stop_mode_lookup(stops, stop_times, trips, routes)
print(stop_mode_lookup.value_counts("mode"))
print(stop_mode_lookup.head())
print(find_nearest_stop((33.7490, -84.3880), stops))
print(preprocess_trips(trips))

def gtfs_time_to_minutes(time_str: str) -> int:
    try:
        parts = time_str.strip().split(':')
        h, m = int(parts[0]), int(parts[1])
        return h * 60 + m
    except (ValueError, AttributeError, IndexError):
        return -1

def get_active_service_ids(calendar_df: pd.DataFrame, day_of_week: int) -> set:
    day_col_map = {0: 'monday', 1: 'tuesday', 2: 'wednesday',3: 'thursday', 4: 'friday', 5: 'saturday', 6: 'sunday'}
    col = day_col_map.get(day_of_week)

    if col not in calendar_df.columns:
        return set()

    active = calendar_df[pd.to_numeric(calendar_df[col], errors='coerce') == 1]
    #return the set of service_ids that are active on the given day
    return set(active['service_id'].tolist())

#converting all GTFS departure times to integer minutes 
stop_times['dep_min'] = stop_times['departure_time'].apply(gtfs_time_to_minutes)
stop_times['stop_sequence'] = pd.to_numeric(stop_times['stop_sequence'],errors='coerce')

#remove any rows where parsing failed
stop_times = stop_times[(stop_times['dep_min'] >= 0) & (stop_times['stop_sequence'].notna())].copy()

#Create a dictionary mapping service_id : set of trip_ids
service_to_trips = (trips.groupby('service_id')['trip_id'].apply(set).to_dict())

#Create a dictionary mapping route_id : route info
route_info = routes.set_index('route_id').to_dict('index')

print(f'Processed {len(stop_times):,} stop_time entries')
print(f'{len(service_to_trips)} service patterns in calendar')

#Sanity check to see trips active on each day of the week
for day_num, day_name in enumerate(['Mon','Tue','Wed','Thu','Fri','Sat','Sun']):
        svc_ids = get_active_service_ids(calendar, day_num)
        trip_count = len(trips[trips['service_id'].isin(svc_ids)])
        print(f"  {day_name}: {len(svc_ids)} service ID(s) → {trip_count:,} trips")
        
#Building a KD data structure to find nearby stops efficiently

#Build a KD-tree over all MARTA stop coordinates for fast spatial queries
def build_stop_index(stops_df: pd.DataFrame) -> tuple:
    stop_coords = stops_df[['stop_lat', 'stop_lon']].values
    tree = cKDTree(stop_coords)
    return tree, stop_coords

def mi_to_degrees(mi: float, lat: float = 33.75) -> float:
    return mi / 69.0

def stops_within_radius(lat: float, lon: float,tree: cKDTree,stops_df: pd.DataFrame,radius_mi: float = WALK_RADIUS_MI) -> pd.DataFrame:
    radius_deg = mi_to_degrees(radius_mi, lat)
    #query_ball_point returns indices of all points within radius_deg
    idxs = tree.query_ball_point([lat, lon], radius_deg)
    #return empty DataFrame if no nearby stops
    if not idxs:
        return pd.DataFrame()
    #get the nearby stops from the stops DF using the indices returned by the KD tree query
    nearby = stops_df.iloc[idxs].copy()
    #compute haversine distance for each nearby stop
    nearby['walk_mi'] = haversine_mi_vectorized(nearby['stop_lat'].values,nearby['stop_lon'].values,lat,lon)
    #filter to true radius 
    nearby = nearby[nearby['walk_mi'] <= radius_mi].copy()
    #sort by walk distance — closest stop first
    return nearby.sort_values('walk_mi').reset_index(drop=True)

def haversine_mi_vectorized(lat1_arr, lon1_arr, lat2: float, lon2: float) -> np.ndarray:
    R = 3958.8  # Earth radius in miles
    lat1 = np.radians(np.asarray(lat1_arr, dtype=float))
    lon1 = np.radians(np.asarray(lon1_arr, dtype=float))
    lat2 = np.radians(float(lat2))
    lon2 = np.radians(float(lon2))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2)
    return R * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

#Build the Index
stop_tree, stop_coords_array = build_stop_index(stops)
print(f'KD-tree built over {len(stops):,} MARTA stops.')

# Sanity check: how many stops are within 400m of Georgia Tech (Midtown work cluster)?
test_lat, test_lon = 33.775, -84.396   # Georgia Tech area
test_nearby = stops_within_radius(test_lat, test_lon, stop_tree, stops)
#add stop type info to the nearby stops for better understanding of results
test_nearby = test_nearby.merge(stop_mode_lookup[['stop_id', 'mode']], on='stop_id', how='left')
print(f'Stops within {WALK_RADIUS_MI}mi of Georgia Tech: {len(test_nearby)}')
if not test_nearby.empty:
    print(test_nearby[['stop_name', 'walk_mi', 'mode', 'stop_id']].head(5).to_string(index=False))
    
def find_transit_options_for_trip(
        origin_lat, origin_lon, dest_lat, dest_lon,
        departure_time,
        tree, stops_df, stop_times_df, trips_df, calendar_df,
        top_n=TOP_N_TRANSIT, walk_radius_mi=WALK_RADIUS_MI,
        time_window_min=TIME_WINDOW_MINUTES):
    dep_mins    = departure_time.hour * 60 + departure_time.minute
    day_of_week = departure_time.weekday()

    #Find walkable stops at both ends
    origin_stops = stops_within_radius(origin_lat, origin_lon, tree, stops_df, walk_radius_mi)
    dest_stops   = stops_within_radius(dest_lat,   dest_lon,   tree, stops_df, walk_radius_mi)
    if origin_stops.empty or dest_stops.empty:
        return []

    origin_stop_ids = set(origin_stops['stop_id'].tolist())
    dest_stop_ids   = set(dest_stops['stop_id'].tolist())

    #Get active GTFS service IDs for this day
    active_service_ids = get_active_service_ids(calendar_df, day_of_week)
    active_trip_ids    = set()
    for svc_id in active_service_ids:
        active_trip_ids.update(service_to_trips.get(svc_id, set()))
    if not active_trip_ids:
        return []

    #Filter stop_times to active trips departing near origin within time window
    lower, upper = dep_mins - time_window_min, dep_mins + time_window_min
    origin_st = stop_times_df[
        (stop_times_df['trip_id'].isin(active_trip_ids)) &
        (stop_times_df['stop_id'].isin(origin_stop_ids)) &
        (stop_times_df['dep_min'] >= lower) &
        (stop_times_df['dep_min'] <= upper)
    ]
    if origin_st.empty:
        return []

    #Find destination stops served by the same trips, after the boarding stop
    candidate_trip_ids = set(origin_st['trip_id'].tolist())
    dest_st = stop_times_df[
        (stop_times_df['trip_id'].isin(candidate_trip_ids)) &
        (stop_times_df['stop_id'].isin(dest_stop_ids))
    ]
    if dest_st.empty:
        return []

    #Join board + deboard rows; enforce forward direction via stop_sequence
    pairs = origin_st[['trip_id','stop_id','stop_sequence','dep_min']].merge(
        dest_st[['trip_id','stop_id','stop_sequence','dep_min']],
        on='trip_id', suffixes=('_board','_deboard'))
    pairs = pairs[pairs['stop_sequence_deboard'] > pairs['stop_sequence_board']]
    if pairs.empty:
        return []

    trip_to_route = trips_df.set_index('trip_id')['route_id'].to_dict()
    results = []

    for _, row in pairs.iterrows():
        trip_id       = row['trip_id']
        board_stop_id = row['stop_id_board']
        deboard_stop_id = row['stop_id_deboard']
        board_dep_min   = row['dep_min_board']
        deboard_dep_min = row['dep_min_deboard']

        board_stop   = origin_stops[origin_stops['stop_id'] == board_stop_id]
        deboard_stop = dest_stops[dest_stops['stop_id'] == deboard_stop_id]
        if board_stop.empty or deboard_stop.empty:
            continue
        board_stop   = board_stop.iloc[0]
        deboard_stop = deboard_stop.iloc[0]

        walk_to_min   = (board_stop['walk_mi']   / WALK_SPEED_MI_H) * 60
        ride_min      = max(deboard_dep_min - board_dep_min, 0)
        walk_from_min = (deboard_stop['walk_mi'] / WALK_SPEED_MI_H) * 60
        total_min     = walk_to_min + ride_min + walk_from_min
        if total_min <= 0:
            continue

        dist_mi   = float(haversine_mi_vectorized([origin_lat],[origin_lon],dest_lat,dest_lon)[0])
        drive_min = (dist_mi * 1.3 / AVG_DRIVE_SPEED_MI_H) * 60
        time_ratio = total_min / max(drive_min, 1)

        time_delta    = abs(board_dep_min - dep_mins)
        schedule_fit  = 1.0 - (time_delta / time_window_min)
        walk_score    = 1.0 - min(board_stop['walk_mi'] / walk_radius_mi, 1.0)
        time_ratio_score = max(0.0, 1.0 - max(time_ratio - 1.0, 0.0))
        score = 0.45*schedule_fit + 0.30*walk_score + 0.25*time_ratio_score

        route_id   = trip_to_route.get(trip_id, '')
        rinfo      = route_info.get(route_id, {})
        route_type = rinfo.get('route_type_text', rinfo.get('route_type', 'Bus'))

        co2_drive   = dist_mi * 1.3 * CO2_DRIVE_LB_PER_MI
        co2_factor  = CO2_RAIL_LB_PER_MI if 'rail' in str(route_type).lower() else CO2_BUS_LB_PER_MI
        co2_transit = dist_mi * 1.0 * co2_factor
        co2_saved   = round(co2_drive - co2_transit, 3)

        results.append({
            'route_id':          route_id,
            'route_short_name':  rinfo.get('route_short_name', route_id),
            'route_long_name':   rinfo.get('route_long_name', ''),
            'route_type':        str(route_type),
            'route_color':       rinfo.get('route_color', '008000'),
            'board_stop_name':   board_stop['stop_name'],
            'board_stop_id':     board_stop_id,
            'deboard_stop_name': deboard_stop['stop_name'],
            'deboard_stop_id':   deboard_stop_id,
            'departure_time':    f"{board_dep_min//60:02d}:{board_dep_min%60:02d}",
            'arrival_time':      f"{deboard_dep_min//60:02d}:{deboard_dep_min%60:02d}",
            'time_delta_min':    round(time_delta, 1),
            'walk_to_stop_min':  round(walk_to_min, 1),
            'walk_to_stop_mi':   round(board_stop['walk_mi'], 3),
            'ride_min':          round(ride_min, 1),
            'walk_from_stop_min':round(walk_from_min, 1),
            'walk_from_stop_mi': round(deboard_stop['walk_mi'], 3),
            'total_min':         round(total_min, 1),
            'drive_min_estimate':round(drive_min, 1),
            'time_vs_driving':   f"+{round(total_min-drive_min,0):.0f} min"
                                 if total_min > drive_min
                                 else f"{round(total_min-drive_min,0):.0f} min (faster!)",
            'co2_saved_lbs':     round(co2_saved * 2.205, 2),
            'score':             round(score, 4),
            'schedule_fit':      round(schedule_fit, 4),
            'walk_score':        round(walk_score, 4),
            'time_ratio':        round(time_ratio, 3),
        })

    if not results:
        return []

    #Deduplicate: keep best-scoring departure per route line
    seen_routes = {}
    for r in sorted(results, key=lambda x: x['score'], reverse=True):
        key = r['route_short_name']
        if key not in seen_routes:
            seen_routes[key] = r
    return list(seen_routes.values())[:top_n]

print('Core transit matching function defined.')

def get_transit_suggestions_for_user(user_id: int,
                                      all_trips: pd.DataFrame,
                                      friendships_dict: dict) -> list:
    #Step 1: Get this user's recurring trips via DBSCAN 
    my_trips = all_trips[all_trips['user_id'] == user_id]
    my_recurring = get_recurring_trips(my_trips)

    if my_recurring.empty:
        print(f'User {user_id}: no recurring trips found.')
        return []

    #Get the distinct OD cluster pairs this user travels recurringly
    my_od_pairs = my_recurring[['start_cluster', 'end_cluster']].drop_duplicates()

    #Step 2: Get this user's direct friends
    my_friends = friendships_dict.get(user_id, set())

    all_suggestions = []

    #Step 3: For each recurring OD pair, find transit + matching friends 
    for _, od in my_od_pairs.iterrows():
        sc = od['start_cluster']
        ec = od['end_cluster']

        #Get the canonical (median) trip for this OD pair
        canonical = get_canonical_trip_for_od(my_recurring, sc, ec)
        pattern_label = f'OD({sc}→{ec})'

        #Step 3a: Find friends who share this OD cluster pair 
        # A friend 'shares' the pattern if they also have recurring trips in
        # the same start_cluster AND end_cluster. This is the same logic used
        # in the carpool matching notebook to identify compatible trip pairs.
        friends_on_route = []
        for friend_id in my_friends:
            friend_trips    = all_trips[all_trips['user_id'] == friend_id]
            friend_recurring = get_recurring_trips(friend_trips)
            if friend_recurring.empty:
                continue
            # Check if this friend has the same OD cluster pair
            friend_od = friend_recurring[['start_cluster','end_cluster']].drop_duplicates()
            match = friend_od[
                (friend_od['start_cluster'] == sc) &
                (friend_od['end_cluster']   == ec)
            ]
            if not match.empty:
                friends_on_route.append(int(friend_id))

        #Step 3b: Find MARTA options for this canonical trip
        options = find_transit_options_for_trip(
            origin_lat     = canonical['start_lat'],
            origin_lon     = canonical['start_lon'],
            dest_lat       = canonical['end_lat'],
            dest_lon       = canonical['end_lon'],
            departure_time = canonical['start_time'],
            tree           = stop_tree,
            stops_df       = stops,
            stop_times_df  = stop_times,
            trips_df       = trips,
            calendar_df    = calendar
        )

        #Attach metadata to each option and collect
        for opt in options:
            opt['pattern_label'] = pattern_label
            opt['user_trip_id'] = int(canonical['trip_id'])
            opt['user_id'] = user_id
            opt['friends_on_route'] = friends_on_route
            all_suggestions.append(opt)

    return all_suggestions

print('deSOVer transit suggestion function defined.')